In [ ]:
!pip install cartopy
!pip install dask
!pip install xarray==2022.11.0
!pip install rioxarray geopandas
!pip install matplotlib
!pip install netCDF4
!pip install optuna
!pip install pysal
!pip install skillmetrics
!pip install alphashape
!pip install shapely
!pip install pygrib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.7/963.7 kB 19.0 MB/s eta 0:00:00
  Attempting uninstall: xarray
    Found existing installation: xarray 2025.12.0
    Uninstalling xarray-2025.12.0:
      Successfully uninstalled xarray-2025.12.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
arviz 0.22.0 requires xarray>=2023.7.0, but you have xarray 2022.11.0 which is incompatible.
xarray-einstats 0.11.0 requires xarray>=2024.07.0, but you have xarray 2022.11.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.8 MB/s eta 0:00:00
  Attempting uninstall: xarray
    Found existing installation: xarray 2022.11.0
    Uninstalling xarray-2022.11.0:
      Successfully uninstalled xarr

In [ ]:
from netCDF4 import Dataset
import numpy as np
import dask.dataframe as dd
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import glob
import matplotlib.patches as patches
from tensorflow.keras.models import load_model

import cartopy.crs as ccrs
from shapely.geometry import mapping
import geopandas as gpd
import os
import cartopy.feature as cfeature
import tensorflow as tf
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.layers import SpatialDropout3D
from tensorflow.keras.layers import (
    SpatialDropout3D, BatchNormalization, Conv3D, MaxPooling3D,
    ConvLSTM2D, Flatten, Dense, Reshape, Input
)
from tensorflow.keras.layers import Layer
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from tensorflow.keras.layers import (
    Input, Conv3D, ConvLSTM2D, BatchNormalization, Activation,
    MaxPooling3D, SpatialDropout3D, Add, Flatten, Dense, Reshape
)
import json
from tensorflow.keras.models import Model
tf.keras.mixed_precision.set_global_policy('mixed_float16')

#from tensorflow.keras.mixed_precision import set_global_policy
#set_global_policy('float32')
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.projections import PolarAxes
import mpl_toolkits.axisartist.grid_finder as gf
import mpl_toolkits.axisartist.floating_axes as fa

Mounted at /content/drive


In [ ]:
import os
import requests
import pygrib
import numpy as np

from netCDF4 import Dataset, date2num
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed


class GFS:
    """
    Baixa o GFS 0.25 (rodada 00 UTC) com uma série de tempo contínua,
    de 6 em 6 horas, sem tempos repetidos:

      anteontem (rodada 00z) -> leads 6, 12, 18, 24
      ontem     (rodada 00z) -> leads 6, 12, 18, 24
      hoje      (rodada 00z) -> leads 6 ... horizonte_h

    Cada rodada cobre só as 24 h seguintes, e a rodada seguinte
    continua de onde a anterior parou. Assim:
      - não há validades duplicadas entre rodadas;
      - não se usa o f000 (análise), que não tem precipitação,
        então todos os arquivos têm 'tp'.

    Com horizonte_h=48 (padrão), a rodada de hoje cobre hoje de
    06 a 18 UTC, amanhã inteiro (00 a 18 UTC) e depois de amanhã
    às 00 UTC (= amanhã 21h em Brasília). Use 240 para 10 dias.
    """

    def __init__(self, reference_time=None, horizonte_h=48,
                 output_base="/content/dataset"):

        # ====================================================
        # CONFIGURAÇÃO
        # ====================================================

        self.base_url = "https://noaa-gfs-bdp-pds.s3.amazonaws.com"

        self.leads_passado = [6, 12, 18, 24]
        self.leads_hoje = list(range(6, horizonte_h + 1, 6))

        self.lon_range = [-85, -30]
        self.lat_range = [-50, 15]

        # Nº de arquivos baixados em paralelo
        self.max_workers = 8

        # ----------------------------------------------------
        # (nome_saida, shortName_pygrib, nivel, PADRÃO_IDX)
        # ----------------------------------------------------
        self.variables = [

            ("u10",  "10u",   None, ":UGRD:10 m above ground:"),
            ("v10",  "10v",   None, ":VGRD:10 m above ground:"),

            ("prmsl", "prmsl", None, ":PRMSL:mean sea level:"),
            ("cape",  "cape",  None, ":CAPE:surface:"),
            ("tp",    "tp",    None, ":APCP:surface:"),

            ("t2m",  "2t",    None, ":TMP:2 m above ground:"),
            ("td2m", "2d",    None, ":DPT:2 m above ground:"),

            ("ur600", "r", 600, ":RH:600 mb:"),

            ("u_975",  "u", 975,  ":UGRD:975 mb:"),
            ("v_975",  "v", 975,  ":VGRD:975 mb:"),
            ("u_1000", "u", 1000, ":UGRD:1000 mb:"),
            ("v_1000", "v", 1000, ":VGRD:1000 mb:"),
            ("u_250",  "u", 250,  ":UGRD:250 mb:"),
            ("v_250",  "v", 250,  ":VGRD:250 mb:"),
            ("u_450",  "u", 450,  ":UGRD:450 mb:"),
            ("v_450",  "v", 450,  ":VGRD:450 mb:"),
            ("u_850",  "u", 850,  ":UGRD:850 mb:"),
            ("v_850",  "v", 850,  ":VGRD:850 mb:"),

            ("w500", "w", 500, ":VVEL:500 mb:"),

            ("t850", "t", 850, ":TMP:850 mb:"),
            ("t500", "t", 500, ":TMP:500 mb:"),
            ("t700", "t", 700, ":TMP:700 mb:"),

            ("ur850", "r", 850, ":RH:850 mb:"),
            ("ur500", "r", 500, ":RH:500 mb:"),
            ("ur700", "r", 700, ":RH:700 mb:"),
        ]

        self.idx_patterns = [v[3] for v in self.variables]

        # ====================================================
        # DATAS (somente rodada 00 UTC)
        # ====================================================

        if reference_time is None:
            reference_time = datetime.now(timezone.utc)

        if reference_time.tzinfo is None:
            reference_time = reference_time.replace(tzinfo=timezone.utc)
        else:
            reference_time = reference_time.astimezone(timezone.utc)

        hoje = datetime(
            reference_time.year, reference_time.month, reference_time.day,
            tzinfo=timezone.utc,
        )

        self.reference_time = reference_time
        self.dates = [hoje - timedelta(days=2), hoje - timedelta(days=1), hoje]

        self.leads_by_date = {
            self.dates[0]: self.leads_passado,
            self.dates[1]: self.leads_passado,
            self.dates[2]: self.leads_hoje,
        }

        # ====================================================
        # DIRETÓRIO / SESSÃO
        # ====================================================

        # Caminho absoluto: download e processamento sempre na mesma pasta
        self.output_base = output_base

        self.session = requests.Session()
        self.session.headers.update({"User-Agent": "Mozilla/5.0"})

    # ========================================================
    # ÍNDICE (.idx) -> INTERVALOS DE BYTES
    # ========================================================

    def _byte_ranges(self, idx_text):

        entries = []

        for line in idx_text.splitlines():
            if not line.strip():
                continue
            parts = line.split(":")
            entries.append((int(parts[1]), line))

        entries.sort(key=lambda e: e[0])

        ranges = []

        for i, (start, line) in enumerate(entries):
            if any(p in line for p in self.idx_patterns):
                end = entries[i + 1][0] - 1 if i + 1 < len(entries) else None
                ranges.append([start, end])

        faltando = [
            p for p in self.idx_patterns
            if not any(p in line for _, line in entries)
        ]
        if faltando:
            print(f"Ausentes no idx: {faltando}")

        ranges.sort(key=lambda r: r[0])

        merged = []

        for r in ranges:
            if (
                merged
                and merged[-1][1] is not None
                and r[0] == merged[-1][1] + 1
            ):
                merged[-1][1] = r[1]
            else:
                merged.append(r)

        return merged

    # ========================================================
    # DOWNLOAD (por byte-range)
    # ========================================================

    def download_one(self, date, fh):

        date_str = date.strftime("%Y%m%d")
        fh_str = f"{fh:03d}"

        grib_dir = os.path.join(
            self.output_base, date.strftime("%Y"), date.strftime("%m"), date_str,
        )
        os.makedirs(grib_dir, exist_ok=True)

        filename = os.path.join(grib_dir, f"gfs.t00z.pgrb2.0p25.f{fh_str}")

        if os.path.exists(filename):
            print(f"Já existe: {filename}")
            return filename

        grib_url = (
            f"{self.base_url}/gfs.{date_str}/00/atmos/"
            f"gfs.t00z.pgrb2.0p25.f{fh_str}"
        )
        idx_url = grib_url + ".idx"

        try:

            idx_resp = self.session.get(idx_url, timeout=60)

            if idx_resp.status_code == 404:
                print(f"Ainda não disponível no bucket: {date_str} f{fh_str}")
                return None

            idx_resp.raise_for_status()

            ranges = self._byte_ranges(idx_resp.text)

            if not ranges:
                print(f"Nenhuma variável encontrada no idx: {idx_url}")
                return None

            tmp = filename + ".part"

            with open(tmp, "wb") as f:
                for start, end in ranges:
                    rng = f"bytes={start}-" if end is None else f"bytes={start}-{end}"
                    r = self.session.get(grib_url, headers={"Range": rng}, timeout=120)
                    r.raise_for_status()
                    f.write(r.content)

            os.replace(tmp, filename)

            mb = os.path.getsize(filename) / 1e6
            print(f"Download OK ({mb:.1f} MB): {filename}")

            return filename

        except requests.RequestException as error:
            print(f"Erro no download {date_str} f{fh_str}: {error}")
            return None

    def download(self, date, leads):

        files = []

        with ThreadPoolExecutor(max_workers=self.max_workers) as ex:
            futures = {ex.submit(self.download_one, date, fh): fh for fh in leads}
            for fut in as_completed(futures):
                result = fut.result()
                if result:
                    files.append(result)

        return sorted(files)

    # ========================================================
    # PROCESSAMENTO
    # ========================================================

    def process(self, grib_file):

        print(f"Processando: {grib_file}")

        grbs = pygrib.open(grib_file)

        message = grbs.message(1)
        latitude, longitude = message.latlons()

        lat = latitude[:, 0].copy()
        lon = longitude[0, :].copy()
        lon[lon > 180] -= 360

        lat_idx = np.where((lat >= self.lat_range[0]) & (lat <= self.lat_range[1]))[0]
        lon_idx = np.where((lon >= self.lon_range[0]) & (lon <= self.lon_range[1]))[0]

        lat_min, lat_max = lat_idx[0], lat_idx[-1]
        lon_min, lon_max = lon_idx[0], lon_idx[-1]

        data = {}

        for name, short_name, level, _pattern in self.variables:
            try:
                if short_name == "tp":
                    # Alguns prazos têm dois acumulados (últimas 6 h e desde o
                    # início da rodada). Fica sempre com o de 6 h, para que
                    # 'tp' signifique a mesma coisa em todos os arquivos.
                    msgs = grbs.select(shortName="tp")
                    seis_h = [
                        m for m in msgs
                        if int(m["endStep"]) - int(m["startStep"]) == 6
                    ]
                    message = (seis_h or msgs)[0]
                elif level is not None:
                    message = grbs.select(shortName=short_name, level=level)[0]
                else:
                    message = grbs.select(shortName=short_name)[0]

                data[name] = message.values[lat_min:lat_max + 1, lon_min:lon_max + 1]

            except (IndexError, KeyError, ValueError) as error:
                print(f"{name}: {error}")

        grbs.close()

        return lat[lat_min:lat_max + 1], lon[lon_min:lon_max + 1], data

    # ========================================================
    # SALVAR NETCDF (com tempo)
    # ========================================================

    def save(self, output_file, lat, lon, data, run_date, fh):

        valid = run_date + timedelta(hours=fh)
        units = "hours since 1970-01-01 00:00:00"

        with Dataset(output_file, "w", format="NETCDF4") as nc:

            nc.createDimension("time", None)
            nc.createDimension("latitude", len(lat))
            nc.createDimension("longitude", len(lon))

            t = nc.createVariable("time", "f8", ("time",))
            t.units = units
            t.calendar = "standard"
            t.standard_name = "time"
            t.long_name = "valid time"
            t[:] = [date2num(valid.replace(tzinfo=None), units)]

            rt = nc.createVariable("reference_time", "f8")
            rt.units = units
            rt.calendar = "standard"
            rt.standard_name = "forecast_reference_time"
            rt.assignValue(date2num(run_date.replace(tzinfo=None), units))

            st = nc.createVariable("step", "i4")
            st.units = "hours"
            st.standard_name = "forecast_period"
            st.assignValue(fh)

            la = nc.createVariable("latitude", "f4", ("latitude",))
            la.units = "degrees_north"
            la[:] = lat

            lo = nc.createVariable("longitude", "f4", ("longitude",))
            lo.units = "degrees_east"
            lo[:] = lon

            for name, values in data.items():
                v = nc.createVariable(name, "f4", ("time", "latitude", "longitude"))
                v.coordinates = "reference_time step"
                if name == "tp":
                    v.long_name = "precipitação acumulada nas 6 h anteriores"
                    v.units = "kg m-2"
                v[0, :, :] = values

            nc.title = "GFS 0.25 - recorte América do Sul"
            nc.run = f"{run_date:%Y-%m-%d} 00 UTC"
            nc.forecast_hour = fh

    # ========================================================
    # EXECUÇÃO
    # ========================================================

    def run(self):

        faltando = []

        for date in self.dates:

            date_str = date.strftime("%Y%m%d")
            leads = self.leads_by_date[date]

            print("\n" + "=" * 60)
            print(f"GFS: {date_str} 00 UTC ({len(leads)} leads: {leads[0]}h a {leads[-1]}h)")
            print("=" * 60)

            grib_files = self.download(date, leads)

            output_dir = os.path.join(self.output_base, date.strftime("%Y"), date.strftime("%m"))
            os.makedirs(output_dir, exist_ok=True)

            baixados = set()

            for grib_file in grib_files:

                fh_str = os.path.basename(grib_file).split(".")[-1]   # "f030"
                fh = int(fh_str[1:])
                baixados.add(fh)

                output_file = os.path.join(output_dir, f"gfs.{date_str}.00z.{fh_str}.nc")

                if os.path.exists(output_file):
                    print(f"Já processado: {output_file}")
                    continue

                try:
                    lat, lon, data = self.process(grib_file)
                    self.save(output_file, lat, lon, data, date, fh)
                    print(f"Salvo: {output_file}")
                except Exception as error:
                    print(f"Erro ao converter {grib_file}: {error}")
                    if os.path.exists(output_file):
                        os.remove(output_file)
                    faltando.append((date, fh))

            faltando += [(date, fh) for fh in leads if fh not in baixados]

        # ---------------- resumo ----------------
        print("\n" + "=" * 60)
        if faltando:
            print("Prazos que NÃO viraram NetCDF (rode de novo mais tarde):")
            for date, fh in sorted(faltando):
                valid = date + timedelta(hours=fh)
                print(f"  rodada {date:%d/%m} f{fh:03d} -> válido {valid:%d/%m %H} UTC")
        else:
            inicio = self.dates[0] + timedelta(hours=self.leads_passado[0])
            fim = self.dates[-1] + timedelta(hours=self.leads_hoje[-1])
            print(f"Completo: {inicio:%d/%m %H} UTC até {fim:%d/%m %H} UTC, de 6 em 6 h")


# ============================================================
# EXECUTAR
# ============================================================

if __name__ == "__main__":
    GFS().run()


GFS: 20261016 00 UTC (4 leads: 6h a 24h)
Download OK (20.2 MB): /content/dataset/2026/10/20261016/gfs.t00z.pgrb2.0p25.f006
Download OK (20.4 MB): /content/dataset/2026/10/20261016/gfs.t00z.pgrb2.0p25.f018
Download OK (19.4 MB): /content/dataset/2026/10/20261016/gfs.t00z.pgrb2.0p25.f024
Download OK (20.2 MB): /content/dataset/2026/10/20261016/gfs.t00z.pgrb2.0p25.f012
Processando: /content/dataset/2026/10/20261016/gfs.t00z.pgrb2.0p25.f006
Salvo: /content/dataset/2026/10/gfs.20261016.00z.f006.nc
Processando: /content/dataset/2026/10/20261016/gfs.t00z.pgrb2.0p25.f012
Salvo: /content/dataset/2026/10/gfs.20261016.00z.f012.nc
Processando: /content/dataset/2026/10/20261016/gfs.t00z.pgrb2.0p25.f018
Salvo: /content/dataset/2026/10/gfs.20261016.00z.f018.nc
Processando: /content/dataset/2026/10/20261016/gfs.t00z.pgrb2.0p25.f024
Salvo: /content/dataset/2026/10/gfs.20261016.00z.f024.nc

GFS: 20261017 00 UTC (4 leads: 6h a 24h)
Download OK (19.4 MB): /content/dataset/2026/10/20261017/gfs.t00z.pgrb2.

In [ ]:
import xarray as xr
import os
import glob
import numpy as np
from datetime import datetime
import pandas as pd
import re

class ComputerAtmosphericParameters:

    def __init__(self,path_file):
        self.path_file=path_file

    def process_file_to_xarray(self):

      nome = os.path.basename(self.path_file)
      m = re.search(r"gfs\.(\d{8})\.00z\.f(\d{3})\.nc$", nome)
      rodada = pd.Timestamp(m.group(1))
      fh = int(m.group(2))
      validade = rodada + pd.Timedelta(hours=fh)

      ds = xr.open_dataset(self.path_file, engine='netcdf4')
      ds = ds.rename({'latitude': 'lat', 'longitude': 'lon'})

      if 'time' in ds.dims:
          ds = ds.isel(time=0, drop=True)
      ds = ds.drop_vars(['reference_time', 'step'], errors='ignore')

      ds = ds.expand_dims(time=[validade])
      return ds

    @staticmethod
    def wind_shear(u_wind_mean_surf,v_wind_mean_surf,u450,v450):
        return np.sqrt(((u_wind_mean_surf-u450)**2)+(v_wind_mean_surf-v450)**2)

    def dls (self,ds):
        ds['u_mean_sup']=(ds['u10']+ds['u_975'])/2
        ds['v_mean_sup']=(ds['v10']+ds['v_975'])/2
        ds['dls']=self.wind_shear(ds['u_mean_sup'],ds['v_mean_sup'],ds['u_450'],ds['v_450'])

        return ds[['dls']]

    @staticmethod
    def thermal_gradient(ds):
        temp_850 = ds['t850']
        lat = ds['lat'].values  # 1D
        lon = ds['lon'].values  # 1D

        lat_rad = np.deg2rad(lat)
        dlat = np.gradient(lat) * 111.32e3  # 1 grau ≈ 111.32 km
        dlon = np.gradient(lon) * (111.32e3 * np.cos(lat_rad.mean()))  # Ajuste pela latitude média

        dT_dlat, dT_dlon = np.gradient(temp_850, axis=(-2, -1))
        dT_dlat /= dlat[:, None]  # Normaliza dT/dlat
        dT_dlon /= dlon[None, :]  # Normaliza dT/dlon

        # Cria um novo Dataset com os gradientes
        grad_temp = xr.Dataset(
            {
                'dT_dlat': (['time', 'lat', 'lon'], dT_dlat, temp_850.attrs),
                'dT_dlon': (['time', 'lat', 'lon'], dT_dlon, temp_850.attrs),
                'grad_T': (['time', 'lat', 'lon'], np.sqrt(dT_dlat**2 + dT_dlon**2), temp_850.attrs),
            },
            coords={
                'time': ds['time'],
                'lat': ds['lat'],
                'lon': ds['lon']
            }
        )
        return grad_temp

    def calculate_dew_point(self, temp, rh):
        a = 17.27
        b = 237.7  # °C
        # Evitar divisão por zero ou log(0)
        rh_safe = np.maximum(rh, 1e-6)
        alpha = (a * temp) / (b + temp) + np.log(rh_safe / 100.0)
        dew_point = (b * alpha) / (a - alpha)
        return dew_point


    def calculate_tt(self,t850, t500, td850):
        return (t850 - t500) + (td850 - t500)

    def calculate_ki(self,t850, t500, td850, t700, td700):
        return (t850 - t500) + td850 - (t700 - td700)

    @staticmethod
    def calculate_wind_divergence(ds, u_var, v_var):
        u = ds[u_var].values
        v = ds[v_var].values
        lat = ds['lat'].values  # convert to numpy array
        lon = ds['lon'].values  # convert to numpy array
        lat_rad = np.radians(lat)
        lon_rad = np.radians(lon)
        dlat = np.gradient(lat_rad)[:, None]  # now lat_rad is a numpy array
        dlon = np.gradient(lon_rad)[None, :]   # now lon_rad is a numpy array
        R = 6371000
        du_dx = np.gradient(u, axis=-1) / (R * np.cos(lat_rad)[:, None] * dlon)
        dv_dy = np.gradient(v, axis=-2) / (R * dlat)
        div = du_dx + dv_dy
        return xr.DataArray(div, coords=ds.coords, dims=ds.dims)

    def compute_indices_from_netcdf(self):

        ds = self.process_file_to_xarray()
        t850 = ds['t850'] - 273.15
        t500 = ds['t500']- 273.15
        t700 = ds['t700']- 273.15

        td850=self.calculate_dew_point(t850,ds['ur850'])
        td700=self.calculate_dew_point(t700,ds['ur700'])

        ds['TT'] = self.calculate_tt(t850, t500, td850)
        ds['KI'] = self.calculate_ki(t850, t500, td850, t700, td700)
        ds['div850'] = self.calculate_wind_divergence(ds,'u_850','v_850')
        ds['div250'] = self.calculate_wind_divergence(ds,'u_250','v_250')
        ds['lls'] = self.wind_shear(ds['u10'],ds['v10'],ds['u_1000'],ds['v_1000'])
        ds['dls']=self.dls(ds)['dls']
        grad_T_ds = self.thermal_gradient(ds)
        for var in grad_T_ds.data_vars:
            ds[var] = grad_T_ds[var]
        return ds

In [ ]:
import os, re, glob, traceback
import numpy as np
from datetime import datetime, timezone, timedelta

base_dir = '/content/dataset'
output_dir = 'GFS_parametrs_teste'
os.makedirs(output_dir, exist_ok=True)

FORCAR = False                      # True = reprocessa mesmo se a saída já existir
INCLUIR_VALIDADE_NO_NOME = False    # True = gfs.20261004.00z.f030.valid2026100506.nc

padrao = re.compile(r"gfs\.(\d{8})\.00z\.f(\d{3})\.nc$")

# Procura todos os NetCDF gerados pelo download, sem depender da data de hoje
arquivos = []
for path in glob.glob(os.path.join(base_dir, '*', '*', 'gfs.*.00z.f*.nc')):
    m = padrao.search(os.path.basename(path))
    if m:
        rodada = datetime.strptime(m.group(1), '%Y%m%d').replace(tzinfo=timezone.utc)
        fh = int(m.group(2))
        arquivos.append((rodada, fh, path))

arquivos.sort()

hoje_utc = datetime.now(timezone.utc).date()
amanha = hoje_utc + timedelta(days=1)

resumo = {'ok': [], 'pulado': [], 'erro': []}

for rodada, fh, path_file in arquivos:
    validade = rodada + timedelta(hours=fh)
    tag = '  <-- AMANHÃ' if validade.date() == amanha else ''
    print(f"{os.path.basename(path_file)} | válido em {validade:%d/%m %H} UTC{tag}")

    nome = os.path.basename(path_file)
    if INCLUIR_VALIDADE_NO_NOME:
        nome = nome.replace('.nc', f'.valid{validade:%Y%m%d%H}.nc')
    output_file = os.path.join(output_dir, nome)

    if os.path.exists(output_file) and not FORCAR:
        print("  já existe. Pulando...")
        resumo['pulado'].append((nome, validade))
        continue

    try:
        ds = ComputerAtmosphericParameters(path_file).compute_indices_from_netcdf()

        # Grava rodada, prazo e validade dentro do arquivo
        ds = ds.assign_coords(
            run_time=np.datetime64(rodada.strftime('%Y-%m-%dT%H:%M'), 'ns'),
            valid_time=np.datetime64(validade.strftime('%Y-%m-%dT%H:%M'), 'ns'),
        )
        ds.attrs['forecast_hour'] = fh

        ds.to_netcdf(output_file)
        print("  salvo com sucesso.")
        resumo['ok'].append((nome, validade))
    except Exception:
        print("  ERRO:")
        traceback.print_exc()
        resumo['erro'].append((nome, validade))

# ---------------- resumo ----------------
print("\n" + "=" * 50)
for k, itens in resumo.items():
    print(f"{k}: {len(itens)}")
    for nome, v in itens:
        if v.date() == amanha:
            print(f"   (amanhã) {nome}")

esperados = {(hoje_utc.strftime('%Y%m%d'), h) for h in [30, 36, 42, 48]}
achados = {(r.strftime('%Y%m%d'), fh) for r, fh, _ in arquivos}
faltando = sorted(esperados - achados)
if faltando:
    print(f"\nNetCDF de amanhã NÃO encontrados em {base_dir}: "
          f"{[f'f{h:03d}' for _, h in faltando]}")
    print("-> o problema está no download/conversão, não neste processamento.")

gfs.20261016.00z.f006.nc | válido em 16/10 06 UTC
  salvo com sucesso.
gfs.20261016.00z.f012.nc | válido em 16/10 12 UTC
  salvo com sucesso.
gfs.20261016.00z.f018.nc | válido em 16/10 18 UTC
  salvo com sucesso.
gfs.20261016.00z.f024.nc | válido em 17/10 00 UTC
  salvo com sucesso.
gfs.20261017.00z.f006.nc | válido em 17/10 06 UTC
  salvo com sucesso.
gfs.20261017.00z.f012.nc | válido em 17/10 12 UTC
  salvo com sucesso.
gfs.20261017.00z.f018.nc | válido em 17/10 18 UTC
  salvo com sucesso.
gfs.20261017.00z.f024.nc | válido em 18/10 00 UTC
  salvo com sucesso.
gfs.20261018.00z.f006.nc | válido em 18/10 06 UTC
  salvo com sucesso.
gfs.20261018.00z.f012.nc | válido em 18/10 12 UTC
  salvo com sucesso.
gfs.20261018.00z.f018.nc | válido em 18/10 18 UTC
  salvo com sucesso.
gfs.20261018.00z.f024.nc | válido em 19/10 00 UTC  <-- AMANHÃ
  salvo com sucesso.
gfs.20261018.00z.f030.nc | válido em 19/10 06 UTC  <-- AMANHÃ
  salvo com sucesso.
gfs.20261018.00z.f036.nc | válido em 19/10 12 UTC  <-

In [ ]:
ds=xr.open_mfdataset('/content/GFS_parametrs_teste/*.nc')
rename = {
        'prmsl': 'msl',
        'KI': 'kx',
        'TT': 'totalx',
        'div850': 'div_850',
        'div250': 'div_250',
        'ur850': 'ur_850',
        'ur600': 'ur_600',
        'td2m': 'd2m',
        'ur700': 'ur_700',
        'w500': 'w'
    }

variables = [
    'cape', 'msl', 'div_250', 'div_850',
    'kx', 'totalx', 'ur_600', 'ur_700', 'w',
    'ur_850', 'dls', 'lls', 'd2m',
    't2m', 'grad_T'
]


ds=ds.rename(rename)
ds_synoptic=ds[variables].copy()


/tmp/ipykernel_2713/2157211331.py:1: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  ds=xr.open_mfdataset('/content/GFS_parametrs_teste/*.nc')


In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
import xarray as xr

from tensorflow.keras.models import load_model

class Cast(tf.keras.layers.Layer):
    def call(self, inputs):
        return tf.cast(inputs, tf.float32)

model = load_model(
    '/content/drive/MyDrive/Doutorado/dataset_shapefile_buffer/Bacia_do_Prata_buffered/results/train_data/classification/classification_model_Bacia_do_Prata_buffered_6H.h5',
    compile=False,
    custom_objects={'Cast': Cast}
)
# ----------------------------------------
# Funções auxiliares (iguais às do treino)
# ----------------------------------------
def create_rolling_window(data, labels, window_size):
    num_samples = len(data) - window_size
    rolling_data = [data[i:i+window_size] for i in range(num_samples)]
    rolling_labels = labels[window_size:]
    return np.array(rolling_data), rolling_labels

def fill_nans_in_blocks(data, block_size=100):
    num_blocks = len(data) // block_size
    for i in range(num_blocks + 1):
        start_idx = i * block_size
        end_idx = min((i + 1) * block_size, len(data))
        data[start_idx:end_idx] = np.nan_to_num(data[start_idx:end_idx], nan=0.0)
    return data


def predict_classification_model(ds_new, shapefile_name, variables, steps='6H', window_size=3):
    """
    Usa o modelo treinado para gerar previsões a partir de novos dados (ds_new).

    A janela temporal é definida pelo próprio modelo (model.input_shape[1]),
    já que cada região pode ter sido treinada com um window_size diferente.
    """
    print("🔹 Preparando novos dados de entrada...")

    # Caminho dos arquivos salvos do modelo
    path = f'/content/drive/MyDrive/Doutorado/Model-Trained/{shapefile_name}/results/train_data/classification'
    os.makedirs(path, exist_ok=True)

    if shapefile_name in ['Amazonia_norte_buffered','Amazonia_sul_buffered',
                          'Regiao_Equatorial_buffered','Nordeste_Brasileiro_buffered','Minas_Gerais_Sudeste_Brasileiro_buffered','Sudeste_Brasileiro_Uruguai_Oeste_Argentino_buffered']:
      path = f'/content/drive/MyDrive/Doutorado/Model-Trained/{shapefile_name}/results/train_data/classification'
      model=load_model(f'{path}/nw_classification_model_{shapefile_name}_{steps}.h5',compile=False,custom_objects={'Cast': Cast})
      mean=np.load(f'{path}/nw_mean_{shapefile_name}_{steps}.npy')
      std=np.load(f'{path}/nw_std_{shapefile_name}_{steps}.npy')

    else:

      # Carregar modelo e normalização
      print("🔹 Carregando modelo treinado e normalização...")
      model = load_model(f'{path}/nw_classification_model_{shapefile_name}_{steps}.h5', compile=False)
      mean = np.load(f'{path}/nw_mean_{shapefile_name}_{steps}.npy')
      std = np.load(f'{path}/nw_std_{shapefile_name}_{steps}.npy')

    # ============================================================
    # Janela definida pelo modelo (cada região pode ter a sua)
    # ============================================================
    _shp = model.input_shape
    if isinstance(_shp, list):        # modelo com múltiplas entradas
        _shp = _shp[0]
    _model_ws = _shp[1]

    if _model_ws is not None and _model_ws != window_size:
        print(f"⚠️ window_size {window_size} -> {_model_ws} (ajustado ao modelo)")
        window_size = int(_model_ws)

    print(f"Window size (modelo): {window_size}")

    # Criar arrays de dados
    data = np.stack([ds_new[var].values for var in variables], axis=-1)

    # Se quiser apenas prever (sem labels):
    labels_placeholder = np.zeros_like(data[:, :, :, 0])  # apenas para manter o formato
    data_rolling, _ = create_rolling_window(data, labels_placeholder, window_size)

    # Substituir NaNs
    data_rolling = fill_nans_in_blocks(data_rolling)

    # Normalizar com mean e std do treino
    data_rolling = (data_rolling - mean) / std

    print("🔹 Gerando previsões...")
    predictions = model.predict(data_rolling, batch_size=16, verbose=1)

    print("✅ Previsões concluídas.")
    print(f"  • Shape das previsões: {predictions.shape}")
    print(f"  • Intervalo de valores: {predictions.min():.4f} a {predictions.max():.4f}")

    return predictions


def predict_regressor_model_robust(
    ds_new,
    shapefile_name,
    variables,
    target_name='Flashes',
    steps='6H',
    window_size=5
):
    """
    Gera previsões com o modelo ConvLSTM robusto para TODOS os
    tempos-alvo existentes.

    Convenção de tempo (idêntica ao netcdf_ds e ao classificador):

        janela i = tempos [i, i+window_size-1]  ->  prevê tempo i+window_size
        i varia de 0 até (n_times - window_size - 1)

    Para n tempos, gera (n - window_size) previsões, alinhadas a
    ds_new.time.values[window_size:].

    Saída: preds -> (n_windows, lat, lon)
    """

    # ============================================================
    # 1. Caminhos
    # ============================================================

    base_path = (
        f'/content/drive/MyDrive/Doutorado/'
        f'Model-Trained-new/{shapefile_name}/'
        f'results/train_data/regressor'
    )

    print(f"Carregando modelo e parâmetros de: {base_path}")

    model_path = (
        f'{base_path}/'
        f'regressor_model_{shapefile_name}_{steps}.keras'
    )

    # ============================================================
    # 2. Modelo
    # ============================================================

    model = tf.keras.models.load_model(
        model_path,
        compile=False,
        safe_mode=False
    )

    print("Modelo carregado com sucesso.")

    # ============================================================
    # 3. Estatísticas
    #    np.squeeze remove eixos de tamanho 1 (ex.: batch/time do
    #    treino salvos com keepdims), deixando-as compatíveis com
    #    o array (time, lat, lon, vars).
    # ============================================================

    mean_lbl = np.squeeze(np.load(
        f'{base_path}/mean_regressor_{shapefile_name}_{steps}.npy'
    ))

    std_lbl = np.squeeze(np.load(
        f'{base_path}/std_regressor_{shapefile_name}_{steps}.npy',
        allow_pickle=True
    ))

    mean_data = np.squeeze(np.load(
        f'{base_path}/mean_data_regressor_{shapefile_name}_{steps}.npy'
    ))

    std_data = np.squeeze(np.load(
        f'{base_path}/std_data_regressor_{shapefile_name}_{steps}.npy'
    ))

    print("Shapes stats (entrada):", mean_data.shape, std_data.shape)
    print("Shapes stats (alvo):   ", np.shape(mean_lbl), np.shape(std_lbl))

    # ============================================================
    # 4. Ordenação temporal
    # ============================================================

    ds_new = ds_new.sortby('time')

    n_times = len(ds_new.time)

    print("Número de tempos disponíveis:", n_times)
    print("Tempos:")
    print(ds_new.time.values)

    # ============================================================
    # 5. Dataset -> array  (time, lat, lon, variables)
    # ============================================================

    data_raw = np.stack(
        [ds_new[var].values for var in variables],
        axis=-1
    ).astype(np.float32)

    print("Dados originais:", data_raw.shape)

    # ============================================================
    # 6. Normalização  -> mantém (time, lat, lon, vars)
    # ============================================================

    data_norm = (data_raw - mean_data) / (std_data + 1e-6)
    data_norm = np.asarray(data_norm, dtype=np.float32)

    # Segurança: se algum eixo-fantasma sobrou, volta ao shape original
    if data_norm.shape != data_raw.shape:
        data_norm = data_norm.reshape(data_raw.shape)

    data_norm = np.nan_to_num(
        data_norm,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    print("Dados normalizados:", data_norm.shape)

    # ============================================================
    # 7. Verificação
    # ============================================================

    if n_times <= window_size:
        raise ValueError(
            f"Poucos tempos disponíveis.\n"
            f"Tempos disponíveis: {n_times}\n"
            f"Window size: {window_size}\n"
            f"Necessário: pelo menos {window_size + 1}"
        )

    # ============================================================
    # 8. Janelas deslizantes sobre TODOS os tempos-alvo
    # ============================================================

    n_windows = n_times - window_size

    X_new = np.stack(
        [data_norm[i:i + window_size] for i in range(n_windows)],
        axis=0
    ).astype(np.float32)
    # (n_windows, window_size, lat, lon, vars)

    print("Janelas para previsão:", X_new.shape)

    # ============================================================
    # 9. Previsão
    # ============================================================

    print("🔹 Gerando previsões...")

    preds_norm = model.predict(X_new, verbose=1)

    print("Shape previsão original:", preds_norm.shape)

    # ============================================================
    # 10. Ajustar saída
    # ============================================================

    preds_norm = np.squeeze(preds_norm)

    if preds_norm.ndim == 2:            # caso reste (lat, lon) com 1 janela
        preds_norm = preds_norm[np.newaxis, ...]

    if preds_norm.ndim != 3:
        raise ValueError(
            f"Shape inesperado da previsão: {preds_norm.shape}"
        )

    print("Shape previsão após ajuste:", preds_norm.shape)

    # ============================================================
    # 11. Desnormalização
    # ============================================================

    preds = (preds_norm * std_lbl) + mean_lbl

    # ============================================================
    # 12. Garantir valores >= 0
    # ============================================================

    preds = np.maximum(preds, 0)

    print("Shape final:", preds.shape)
    print(
        f"Intervalo de valores: "
        f"{np.nanmin(preds):.4f} a {np.nanmax(preds):.4f}"
    )

    return preds

def netcdf_ds(
    forecast_data,
    ds_orig,
    window_size=None,   # mantido por compatibilidade; NÃO é mais usado p/ alinhar
    var_name='forecast'
):
    """
    Converte previsões (N, lat, lon) em DataArray.

    As previsões correspondem SEMPRE aos últimos N tempos do dataset,
    independentemente do window_size do modelo:

        N previsões  ->  ds.time[-N:]

    Isso torna o alinhamento robusto a modelos com janelas diferentes.
    """

    ds_orig = ds_orig.sortby('time')

    latitude = ds_orig['lat'].values
    longitude = ds_orig['lon'].values

    forecast_data = np.asarray(forecast_data)

    if forecast_data.ndim == 4 and forecast_data.shape[-1] == 1:
        forecast_data = np.squeeze(forecast_data, axis=-1)

    if forecast_data.ndim != 3:
        raise ValueError(
            f"Shape inesperado do forecast: {forecast_data.shape}\n"
            f"Esperado: (time, lat, lon)"
        )

    n_predictions = forecast_data.shape[0]
    all_times = ds_orig['time'].values

    if n_predictions == 0:
        raise ValueError("Nenhuma previsão gerada (N=0).")

    if n_predictions > len(all_times):
        raise ValueError(
            f"Mais previsões ({n_predictions}) do que tempos "
            f"disponíveis ({len(all_times)})."
        )

    # Alinhamento pelos últimos N tempos
    times = all_times[-n_predictions:]

    # Checagem espacial
    expected_spatial_shape = (len(latitude), len(longitude))
    if forecast_data.shape[1:] != expected_spatial_shape:
        raise ValueError(
            f"Incompatibilidade espacial.\n"
            f"Forecast: {forecast_data.shape[1:]}\n"
            f"Esperado: {expected_spatial_shape}"
        )

    return xr.DataArray(
        forecast_data,
        coords={'time': times, 'lat': latitude, 'lon': longitude},
        dims=['time', 'lat', 'lon'],
        name=var_name
    )


In [ ]:

variables = [
    'cape', 'msl', 'div_250', 'div_850',
    'kx', 'totalx', 'ur_600', 'ur_700', 'w',
    'ur_850', 'dls', 'lls', 'd2m',
    't2m', 'grad_T'
]


# ============================================================
# Configuração das regiões
# ============================================================

region_config = {
    "Nordeste_Brasileiro": {
        "color": "darkred",
        "label": "R1"
    },

    "Sudeste_Brasileiro_Uruguai_Oeste_Argentino": {
        "color": "darkblue",
        "label": "R2"
    },

    "Amazonia_norte": {
        "color": "darkgreen",
        "label": "R3"
    },

    "Amazonia_sul": {
        "color": "lightgreen",
        "label": "R4"
    },

    "Bacia_do_Prata": {
        "color": "purple",
        "label": "R5"
    },

    "Minas_Gerais_Sudeste_Brasileiro": {
        "color": "orange",
        "label": "R6"
    },

    "Regiao_Equatorial": {
        "color": "saddlebrown",
        "label": "R7"
    },

    "Colombia_Maracaibo": {
        "color": "violet",
        "label": "R8"
    },
}


def get_region_config(name):

    clean_name = name.replace("_buffered", "")

    return region_config.get(clean_name)


# ============================================================
# Thresholds
# ============================================================

thresholds = {
    "R1": 0.15,
    "R2": 0.15,
    "R3": 0.15,
    "R4": 0.15,
    "R5": 0.15,
    "R6": 0.15,
    "R7": 0.15,
    "R8": 0.15
}


# ============================================================
# Regiões que serão processadas
# ============================================================

regions_to_process = [
    'Colombia_Maracaibo_buffered',
    'Bacia_do_Prata_buffered',
    'Regiao_Equatorial_buffered',
    'Nordeste_Brasileiro_buffered',
    'Minas_Gerais_Sudeste_Brasileiro_buffered',
    'Sudeste_Brasileiro_Uruguai_Oeste_Argentino_buffered',
    'Amazonia_norte_buffered','Amazonia_sul_buffered',
    'Amazonia_norte_buffered','Amazonia_sul_buffered'
]

# ============================================================
# Loop sobre os shapefiles
# ============================================================

for shapefile_new in glob.glob(
    '/content/drive/MyDrive/Doutorado/'
    'shapefiles_clusters_with_buffer/*.shp'
):

    shape_new = os.path.basename(
        shapefile_new
    ).replace('.shp', '')

    # --------------------------------------------------------
    # Verificar se a região deve ser processada
    # --------------------------------------------------------

    if shape_new not in regions_to_process:
        print(
            f"⏭️ Pulando região: {shape_new}"
        )
        continue

    print("\n" + "=" * 70)
    print(f"PROCESSANDO: {shape_new}")
    print("=" * 70)

    # --------------------------------------------------------
    # Configuração da região
    # --------------------------------------------------------

    region_conf = get_region_config(shape_new)

    if region_conf is None:

        print(
            f"❌ Região não encontrada no region_config: "
            f"{shape_new}"
        )

        continue

    label = region_conf['label']

    print(f"Região: {label}")
    print(f"Shapefile: {shape_new}")

    # --------------------------------------------------------
    # Ler shapefile
    # --------------------------------------------------------

    shape = gpd.read_file(
        shapefile_new
    )

    # --------------------------------------------------------
    # Renomear variáveis do GFS
    # --------------------------------------------------------

    ds_gfs_synoptic = ds_synoptic.copy()

    # --------------------------------------------------------
    # Garantir ordem crescente
    # --------------------------------------------------------

    ds_gfs_synoptic = ds_gfs_synoptic.sortby(
        ['lat', 'lon']
    )

    # --------------------------------------------------------
    # Extensão espacial do shapefile
    # --------------------------------------------------------

    minx, miny, maxx, maxy = shape.total_bounds

    print(
        f"Extensão: "
        f"lon={minx:.2f} até {maxx:.2f}, "
        f"lat={miny:.2f} até {maxy:.2f}"
    )

    # --------------------------------------------------------
    # Selecionar domínio espacial
    # --------------------------------------------------------

    ds_t2 = ds_gfs_synoptic.sel(
        lon=slice(minx, maxx),
        lat=slice(miny, maxy)
    )

    # --------------------------------------------------------
    # Dataset final
    # --------------------------------------------------------

    ds = ds_t2.copy()

    ds = ds.sortby('time')

    print(
        f"Número de tempos disponíveis: "
        f"{len(ds.time)}"
    )

    print(
        "Tempos disponíveis:"
    )

    print(ds.time.values)

    # ========================================================
    # Nomes das variáveis de saída
    # ========================================================

    var_prob_name = 'prob-flash-gfs'
    var_regressor_name = 'deter-flash-gfs'

    # ========================================================
    # 1. CLASSIFICAÇÃO
    # ========================================================

    print("\n🔵 CLASSIFICAÇÃO")

    predictions_classification = (
        predict_classification_model(
            ds,
            shape_new,
            variables,
            steps='6H',
            window_size=6
        )
    )

    print(
        "Shape classificação:",
        predictions_classification.shape
    )

    # --------------------------------------------------------
    # Converter classificação para Dataset
    # --------------------------------------------------------

    ds_class = netcdf_ds(
        predictions_classification,
        ds,
        window_size=6,
        var_name=var_prob_name
    ).to_dataset()

    print(
        "Tempos classificação:",
        ds_class.time.values
    )

    # --------------------------------------------------------
    # Aplicar threshold
    # --------------------------------------------------------

    ds_class[var_prob_name] = (
        ds_class[var_prob_name]
        .where(
            ds_class[var_prob_name] > thresholds[label],
            0.0
        )
    )

    # ========================================================
    # 2. REGRESSÃO
    # ========================================================

    print("\n🟢 REGRESSÃO")

    predictions_regressor = (
        predict_regressor_model_robust(
            ds,
            shape_new,
            variables,
            steps='6H',
            window_size=5
        )
    )

    print(
        "Shape previsão regressor:",
        predictions_regressor.shape
    )

    # --------------------------------------------------------
    # Converter regressor para Dataset
    # --------------------------------------------------------

    ds_regressor = netcdf_ds(
        predictions_regressor,
        ds,
        window_size=5,
        var_name=var_regressor_name
    ).to_dataset()

    print(
        "Tempos regressor:",
        ds_regressor.time.values
    )

    # ========================================================
    # 3. ALINHAR CLASSIFICADOR E REGRESSOR
    # ========================================================

    # Como o classificador usa window=6 e o regressor
    # usa window=5, os tempos iniciais podem ser diferentes.
    #
    # Mantemos somente os tempos presentes nos dois.

    common_times = np.intersect1d(
        ds_class.time.values,
        ds_regressor.time.values
    )

    print(
        "\nTempos comuns:",
        common_times
    )

    if len(common_times) == 0:

        print(
            "❌ Nenhum tempo em comum entre "
            "classificador e regressor."
        )

        continue

    # --------------------------------------------------------
    # Selecionar mesmos tempos
    # --------------------------------------------------------

    ds_class = ds_class.sel(
        time=common_times
    )

    ds_regressor = ds_regressor.sel(
        time=common_times
    )

    # ========================================================
    # 4. MERGE
    # ========================================================

    ds_predict_final = xr.merge(
        [
            ds_class,
            ds_regressor
        ],
        join='exact'
    )

    print(
        "\nDataset final:"
    )

    print(
        ds_predict_final
    )

    print(
        "Tempos finais:",
        ds_predict_final.time.values
    )

    # ========================================================
    # 5. Aplicar máscara
    # ========================================================

    mask = (
        ~ds_predict_final[
            var_regressor_name
        ].isnull()
        .all(dim=['lat', 'lon'])
    )

    ds_clean = ds_predict_final.sel(
        time=mask
    )

    # ========================================================
    # 6. Zerar regressor onde probabilidade = 0
    # ========================================================

    ds_clean[var_regressor_name] = (
        ds_clean[var_regressor_name]
        .where(
            ds_clean[var_prob_name] != 0,
            0
        )
    )

    # ========================================================
    # 7. Tipos
    # ========================================================

    ds_clean[var_regressor_name] = (
        ds_clean[var_regressor_name]
        .astype(np.int32)
    )

    ds_clean[var_prob_name] = (
        ds_clean[var_prob_name]
        .astype(np.float32)
    )

    # ========================================================
    # 8. Salvar
    # ========================================================

    output_path = (
        '/content/drive/MyDrive/Doutorado/'
        'teste/Operational_validation-new/GFS/'
        f'{shape_new}/'
    )

    os.makedirs(
        output_path,
        exist_ok=True
    )

    output_file = (
        f'{output_path}'
        f'gfs_prediction_{shape_new}.nc'
    )

    ds_clean.to_netcdf(
        output_file
    )

    print(
        f"\n✅ Arquivo salvo:"
    )

    print(
        output_file
    )

    print(
        "Shape final:",
        ds_clean[var_regressor_name].shape
    )


PROCESSANDO: Nordeste_Brasileiro_buffered
Região: R1
Shapefile: Nordeste_Brasileiro_buffered
Extensão: lon=-54.40 até -34.10, lat=-20.84 até 0.40
Número de tempos disponíveis: 16
Tempos disponíveis:
['2026-10-16T06:00:00.000000000' '2026-10-16T12:00:00.000000000'
 '2026-10-16T18:00:00.000000000' '2026-10-17T00:00:00.000000000'
 '2026-10-17T06:00:00.000000000' '2026-10-17T12:00:00.000000000'
 '2026-10-17T18:00:00.000000000' '2026-10-18T00:00:00.000000000'
 '2026-10-18T06:00:00.000000000' '2026-10-18T12:00:00.000000000'
 '2026-10-18T18:00:00.000000000' '2026-10-19T00:00:00.000000000'
 '2026-10-19T06:00:00.000000000' '2026-10-19T12:00:00.000000000'
 '2026-10-19T18:00:00.000000000' '2026-10-20T00:00:00.000000000']

🔵 CLASSIFICAÇÃO
🔹 Preparando novos dados de entrada...
Window size (modelo): 6
🔹 Gerando previsões...
1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step
✅ Previsões concluídas.
  • Shape das previsões: (10, 85, 81)
  • Intervalo de valores: 0.0000 a 0.9497
Shape classificação: (10, 85, 81)
Te

1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step
✅ Previsões concluídas.
  • Shape das previsões: (13, 52, 46)
  • Intervalo de valores: 0.0000 a 0.0974
Shape classificação: (13, 52, 46)
Tempos classificação: ['2026-10-17T00:00:00.000000000' '2026-10-17T06:00:00.000000000'
 '2026-10-17T12:00:00.000000000' '2026-10-17T18:00:00.000000000'
 '2026-10-18T00:00:00.000000000' '2026-10-18T06:00:00.000000000'
 '2026-10-18T12:00:00.000000000' '2026-10-18T18:00:00.000000000'
 '2026-10-19T00:00:00.000000000' '2026-10-19T06:00:00.000000000'
 '2026-10-19T12:00:00.000000000' '2026-10-19T18:00:00.000000000'
 '2026-10-20T00:00:00.000000000']

🟢 REGRESSÃO
Carregando modelo e parâmetros de: /content/drive/MyDrive/Doutorado/Model-Trained-new/Bacia_do_Prata_buffered/results/train_data/regressor
Modelo carregado com sucesso.
Shapes stats (entrada): (15,) (15,)
Shapes stats (alvo):    () ()
Número de tempos disponíveis: 16
Tempos:
['2026-10-16T06:00:00.000000000' '2026-10-16T12:00:00.000000000'
 '2026-10-16T18:00:00.0000

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
Shape previsão original: (11, 52, 46)
Shape previsão após ajuste: (11, 52, 46)
Shape final: (11, 52, 46)
Intervalo de valores: 0.0000 a 477.8736
Shape previsão regressor: (11, 52, 46)
Tempos regressor: ['2026-10-17T12:00:00.000000000' '2026-10-17T18:00:00.000000000'
 '2026-10-18T00:00:00.000000000' '2026-10-18T06:00:00.000000000'
 '2026-10-18T12:00:00.000000000' '2026-10-18T18:00:00.000000000'
 '2026-10-19T00:00:00.000000000' '2026-10-19T06:00:00.000000000'
 '2026-10-19T12:00:00.000000000' '2026-10-19T18:00:00.000000000'
 '2026-10-20T00:00:00.000000000']

Tempos comuns: ['2026-10-17T12:00:00.000000000' '2026-10-17T18:00:00.000000000'
 '2026-10-18T00:00:00.000000000' '2026-10-18T06:00:00.000000000'
 '2026-10-18T12:00:00.000000000' '2026-10-18T18:00:00.000000000'
 '2026-10-19T00:00:00.000000000' '2026-10-19T06:00:00.000000000'
 '2026-10-19T12:00:00.000000000' '2026-10-19T18:00:00.000000000'
 '2026-10-20T00:00:00.000000000']

Dataset final:
<xarray.Data

In [ ]:
import os
import numpy as np
import xarray as xr


regions_to_process = [
    'Colombia_Maracaibo_buffered',
    'Bacia_do_Prata_buffered',
    'Regiao_Equatorial_buffered',
    'Nordeste_Brasileiro_buffered',
    'Minas_Gerais_Sudeste_Brasileiro_buffered',
    'Sudeste_Brasileiro_Uruguai_Oeste_Argentino_buffered',
    'Amazonia_norte_buffered',
    'Amazonia_sul_buffered',
]

base = '/content/drive/MyDrive/Doutorado/teste/Operational_validation-new/GFS'

VARS = ['prob-flash-gfs', 'deter-flash-gfs']


# ============================================================
# 1. Abrir todos os arquivos existentes
# ============================================================

datasets = []

for area in regions_to_process:

    f = f'{base}/{area}/gfs_prediction_{area}.nc'

    if not os.path.exists(f):
        print(f"⚠️ Não encontrado, pulando: {area}")
        continue

    d = xr.open_dataset(f)[VARS]

    # Arredondar coordenadas para garantir uma grade comum
    # (GFS 0.25° -> múltiplos de 0.25; evita "quase-duplicatas" float
    #  que inflariam a grade união no outer-join)
    d = d.assign_coords(
        lat=np.round(d['lat'].astype('float64'), 3),
        lon=np.round(d['lon'].astype('float64'), 3),
    )

    datasets.append(d)
    print(f"✅ {area}: {dict(d.sizes)}")

if not datasets:
    raise FileNotFoundError("Nenhum arquivo de região foi encontrado.")

print(f"\nRegiões abertas: {len(datasets)}")


# ============================================================
# 2. Combinar tomando o MÁXIMO nas interseções
#    - outer-join monta a grade união (NaN fora de cada região)
#    - max(skipna=True) ignora NaN, pega o maior onde há sobreposição
# ============================================================

combined = xr.concat(datasets, dim='region', join='outer')

merged = combined.max(dim='region', skipna=True)


# ============================================================
# 3. Preencher NaN com 0 e coerência prob/deter
#    - sem cobertura = sem raios = 0
#    - probabilidade = 0 onde o determinístico é 0
# ============================================================

deter = merged['deter-flash-gfs'].fillna(0.0)

prob = merged['prob-flash-gfs'].where(deter != 0, 0.0).fillna(0.0)

merged['deter-flash-gfs'] = deter.astype('float32')   # ou .astype('int32')
merged['prob-flash-gfs'] = prob.astype('float32')


# ============================================================
# 4. Ordenar
# ============================================================

merged = merged.sortby(['time', 'lat', 'lon'])

print("\nDataset combinado:")
print(merged)

print(
    "\nNaN restantes -> prob:",
    int(merged['prob-flash-gfs'].isnull().sum()),
    "| deter:",
    int(merged['deter-flash-gfs'].isnull().sum()),
)


# ============================================================
# 5. Salvar
# ============================================================

out_dir = f'{base}/_MERGED'
os.makedirs(out_dir, exist_ok=True)

out_file = f'{out_dir}/gfs_prediction_ALL_regions_max.nc'
merged.to_netcdf(out_file)

print(f"\n✅ Salvo: {out_file}")

✅ Colombia_Maracaibo_buffered: {'time': 11, 'lat': 26, 'lon': 30}
✅ Bacia_do_Prata_buffered: {'time': 11, 'lat': 52, 'lon': 46}
✅ Regiao_Equatorial_buffered: {'time': 10, 'lat': 71, 'lon': 102}
✅ Nordeste_Brasileiro_buffered: {'time': 10, 'lat': 85, 'lon': 81}
✅ Minas_Gerais_Sudeste_Brasileiro_buffered: {'time': 10, 'lat': 86, 'lon': 91}
✅ Sudeste_Brasileiro_Uruguai_Oeste_Argentino_buffered: {'time': 10, 'lat': 107, 'lon': 87}
✅ Amazonia_norte_buffered: {'time': 10, 'lat': 68, 'lon': 105}
✅ Amazonia_sul_buffered: {'time': 10, 'lat': 75, 'lon': 104}

Regiões abertas: 8

Dataset combinado:
<xarray.Dataset> Size: 3MB
Dimensions:          (time: 11, lat: 206, lon: 188)
Coordinates:
  * time             (time) datetime64[ns] 88B 2026-10-17T12:00:00 ... 2026-1...
  * lat              (lat) float64 2kB -39.5 -39.25 -39.0 ... 11.25 11.5 11.75
  * lon              (lon) float64 2kB -81.0 -80.75 -80.5 ... -34.5 -34.25
Data variables:
    prob-flash-gfs   (time, lat, lon) float32 2MB 0.0 0.0 0.0 

**Suaviza time por time**

In [ ]:
# ============================================================
# SUAVIZAR CADA TEMPO DO GFS E SALVAR EM NETCDF
# ============================================================

import numpy as np
import xarray as xr
from scipy.ndimage import gaussian_filter, median_filter


# ============================================================
# 1. FUNÇÃO DE SUAVIZAÇÃO (2D: lat x lon)
# ============================================================

def suavizar(da, upscale=4, sigma=1.2, despeckle=True,
             vmin=0.0, vmax=None, remask=True):

    da = da.sortby('lat').sortby('lon')
    src = da.values.astype('float64')

    if despeckle:
        src = median_filter(np.nan_to_num(src, nan=0.0), size=3)

    src_da = da.copy(data=src)

    new_lat = np.linspace(float(da.lat.min()), float(da.lat.max()),
                          da.sizes['lat'] * upscale)
    new_lon = np.linspace(float(da.lon.min()), float(da.lon.max()),
                          da.sizes['lon'] * upscale)

    fine = src_da.interp(lat=new_lat, lon=new_lon, method='cubic')

    vals = gaussian_filter(np.nan_to_num(fine.values, nan=0.0), sigma=sigma)

    if vmin is not None:
        vals = np.maximum(vals, vmin)
    if vmax is not None:
        vals = np.minimum(vals, vmax)

    fine = fine.copy(data=vals)

    if remask:
        base0 = src_da.interp(lat=new_lat, lon=new_lon, method='nearest') <= 0
        fine = fine.where(~base0, 0.0)

    return fine


# ============================================================
# 2. SUAVIZAR UMA VARIÁVEL EM TODOS OS TEMPOS
# ============================================================

def suavizar_todos_tempos(da, **kwargs):

    if 'time' not in da.dims:
        return suavizar(da, **kwargs)

    campos = []
    n = da.sizes['time']

    for i in range(n):
        campo = suavizar(da.isel(time=i), **kwargs)
        campos.append(campo)
        print(f'  {da.name}: tempo {i + 1}/{n} '
              f'({str(da.time.values[i])[:16]})')

    return xr.concat(campos, dim='time').assign_coords(time=da.time.values)


# ============================================================
# 3. ABRIR DATASET
# ============================================================

pasta = ('/content/drive/MyDrive/Doutorado/teste/'
         'Operational_validation-new/GFS/_MERGED/')

arquivo = pasta + 'gfs_prediction_ALL_regions_max.nc'
arquivo_saida_nc = pasta + 'gfs_prediction_ALL_regions_suavisado.nc'

ds = xr.open_dataset(arquivo)
print(ds)


# ============================================================
# 4. CONFIGURAÇÃO POR VARIÁVEL
# ============================================================

config = {
    'deter-flash-gfs': dict(upscale=4, sigma=1.2, despeckle=True,
                            vmin=0.0, vmax=None, remask=True),
    'prob-flash-gfs':  dict(upscale=4, sigma=1.2, despeckle=True,
                            vmin=0.0, vmax=1.0, remask=True),
}


# ============================================================
# 5. SUAVIZAR TEMPO A TEMPO
# ============================================================

saida = {}

for var, kw in config.items():

    print(f'\nSuavizando {var}...')

    da_s = suavizar_todos_tempos(ds[var], **kw)
    da_s.attrs = ds[var].attrs.copy()
    da_s.attrs['suavizacao'] = (
        f"median 3x3 + upscale {kw['upscale']}x (cubic) "
        f"+ gaussian sigma={kw['sigma']}"
    )

    saida[var] = da_s

    print(f'  máx original:   {float(ds[var].max()):.4f}')
    print(f'  máx suavizado:  {float(da_s.max()):.4f}')

ds_suavizado = xr.Dataset(saida)
ds_suavizado.attrs = ds.attrs.copy()
ds_suavizado.attrs['fonte'] = 'GFS'
ds_suavizado.attrs['descricao'] = (
    'Campos suavizados tempo a tempo (upscale + gaussian filter) '
    'a partir do gfs_prediction_ALL_regions_max.nc'
)


# ============================================================
# 6. SALVAR (com compressão)
# ============================================================

encoding = {v: {'zlib': True, 'complevel': 4, 'dtype': 'float32'}
            for v in ds_suavizado.data_vars}

ds_suavizado.to_netcdf(arquivo_saida_nc, encoding=encoding)

print('\nArquivo NetCDF salvo em:')
print(arquivo_saida_nc)
print(ds_suavizado)

<xarray.Dataset> Size: 3MB
Dimensions:          (time: 11, lat: 206, lon: 188)
Coordinates:
  * time             (time) datetime64[ns] 88B 2026-10-17T12:00:00 ... 2026-1...
  * lat              (lat) float64 2kB -39.5 -39.25 -39.0 ... 11.25 11.5 11.75
  * lon              (lon) float64 2kB -81.0 -80.75 -80.5 ... -34.5 -34.25
Data variables:
    prob-flash-gfs   (time, lat, lon) float32 2MB ...
    deter-flash-gfs  (time, lat, lon) float32 2MB ...

Suavizando deter-flash-gfs...
  deter-flash-gfs: tempo 1/11 (2026-10-17T12:00)
  deter-flash-gfs: tempo 2/11 (2026-10-17T18:00)
  deter-flash-gfs: tempo 3/11 (2026-10-18T00:00)
  deter-flash-gfs: tempo 4/11 (2026-10-18T06:00)
  deter-flash-gfs: tempo 5/11 (2026-10-18T12:00)
  deter-flash-gfs: tempo 6/11 (2026-10-18T18:00)
  deter-flash-gfs: tempo 7/11 (2026-10-19T00:00)
  deter-flash-gfs: tempo 8/11 (2026-10-19T06:00)
  deter-flash-gfs: tempo 9/11 (2026-10-19T12:00)
  deter-flash-gfs: tempo 10/11 (2026-10-19T18:00)
  deter-flash-gfs: tempo 11